# Fundamentos de Ingeniería de Datos
## EXTRACCIÓN: de la capa RAW a la capa EXTRACTED

```
INGESTA (anterior)                      EXTRACCIÓN (este notebook)
Kaggle → CSV → SQLite: raw_autos   →    extracted_autos
                                        + control de cambios : Logs/cdc_autos.json
                                        + log de auditoría   : Logs/extraccion_log.jsonl
```

### ¿Qué es la capa EXTRACTED?

Son **los mismos datos de `raw_autos`**, sin limpiar todavía, con **4 columnas de metadatos** que dicen cuándo, de dónde y en qué lote llegó cada fila.

| Capa | Tabla | Qué contiene |
|---|---|---|
| RAW | `raw_autos` | Datos originales, tal como llegaron (no se modifica) |
| **EXTRACTED** | **`extracted_autos`** | **Los mismos datos + metadatos de extracción** |
| TRANSFORMED | `transformed_autos.parquet` | Datos limpios (siguiente notebook) |

> **Regla de oro:** cada capa lee solo de la anterior. `raw_autos` se lee en **modo solo lectura**.

### Configuración del entorno

In [1]:
# ── Importar utils.py ──────────────────────────────────────────────────────

import sys
import os
sys.path.append(os.path.abspath('../Utilidades'))   # el notebook se ejecuta desde Codigos/

from utils import (
    PROYECTO_BASE, DB_PATH, EXTRACCION_LOGS, EXPORTS_PATH,
    inspeccionar_dataset, listar_tablas, tamano_mb, ruta_relativa
)

print()
print('utils.py importado correctamente')
print(f'Proyecto base: {os.path.basename(PROYECTO_BASE)}/')
print()


utils.py importado correctamente
Proyecto base: ING_BD_ProjectF/



In [2]:
# ── Librerías, tablas y archivos de esta capa ─────────────────────────────

import json
import hashlib
import sqlite3
import pandas as pd
from datetime import datetime

pd.set_option('display.max_columns', None)


ETIQUETA        = 'autos'
TABLA_RAW       = f'raw_{ETIQUETA}'                              # origen
TABLA_EXTRACTED = f'extracted_{ETIQUETA}'                        # destino

CDC_PATH = f'{EXTRACCION_LOGS}/cdc_{ETIQUETA}.json'              # estado del control de cambios
LOG_PATH = f'{EXTRACCION_LOGS}/extraccion_log.jsonl'             # historial de operaciones

# Columnas para comparar cómo conserva los tipos cada formato (PASO 7)
COLS_TIPOS = ['year', 'price', 'accidents_or_damage', '_batch_id', '_extracted_at']

os.makedirs(EXTRACCION_LOGS, exist_ok=True)
os.makedirs(EXPORTS_PATH, exist_ok=True)

print()
print(f'Origen       : {TABLA_RAW}')
print(f'Destino      : {TABLA_EXTRACTED}')
print(f'CDC          : {ruta_relativa(CDC_PATH)}')
print(f'Log          : {ruta_relativa(LOG_PATH)}')
print(f'Exportación  : {ruta_relativa(EXPORTS_PATH)}')
print()


Origen       : raw_autos
Destino      : extracted_autos
CDC          : Extraccion_de_Datos/Logs/cdc_autos.json
Log          : Extraccion_de_Datos/Logs/extraccion_log.jsonl
Exportación  : Extraccion_de_Datos/Exportaciones



---
## PASO 1 — Revisar el origen

Antes de extraer, verificamos qué tablas existen en el `.db` y leemos `raw_autos` en **modo solo lectura**.

In [3]:
# ── ¿Qué tablas hay en el .db? ─────────────────────────────────────────────

listar_tablas(DB_PATH);


  Tablas en: base_datos.db

  TABLA                               CAPA               FILAS   COLS
  ----------------------------------- --------------- -------- ------


  raw_autos                           raw_             200,000     20


In [4]:
# ── Leer raw_autos en modo solo lectura ────────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)
df_raw = pd.read_sql_query(f'SELECT * FROM {TABLA_RAW}', conn)
conn.close()

inspeccionar_dataset(df_raw, nombre=TABLA_RAW)


  DATASET: raw_autos
  Filas    : 200,000
  Columnas : 20

  Tipos de datos y valores nulos:
    manufacturer                 str          nulos: 0 (0.0%)
    model                        str          nulos: 0 (0.0%)
    year                         int64        nulos: 0 (0.0%)
    mileage                      float64      nulos: 126 (0.1%)
    engine                       str          nulos: 3887 (1.9%)
    transmission                 str          nulos: 2545 (1.3%)
    drivetrain                   str          nulos: 5593 (2.8%)
    fuel_type                    str          nulos: 5977 (3.0%)
    mpg                          str          nulos: 37071 (18.5%)
    exterior_color               str          nulos: 2355 (1.2%)
    interior_color               str          nulos: 15031 (7.5%)
    accidents_or_damage          float64      nulos: 6332 (3.2%)
    one_owner                    float64      nulos: 8265 (4.1%)
    personal_use_only            float64      nulos: 6504 (3.3%)
   

,manufacturer,model,year,mileage,engine,transmission,drivetrain,fuel_type,mpg,exterior_color,interior_color,accidents_or_damage,one_owner,personal_use_only,seller_name,seller_rating,driver_rating,driver_reviews_num,price_drop,price
0,Acura,ILX Hybrid 1.5L,2013,92945.0,"1.5L I-4 i-VTEC variable valve control, engine...",Automatic,Front-wheel Drive,Gasoline,39-38,Black,Parchment,0.0,0.0,0.0,Iconic Coach,NaN,4.4,12.0,300.0,13988.0
1,Acura,ILX Hybrid 1.5L,2013,61539.0,1.5L I4 8V MPFI SOHC Hybrid,Automatic CVT,Front-wheel Drive,Hybrid,39-38,Silver,NaN,1.0,1.0,1.0,Napleton's Palatine Subaru,4.6,4.4,12.0,NaN,16950.0
2,Acura,NSX Base,2017,15904.0,Twin Turbo Gas/Electric V-6 3.5 L/213,Automatic,All-wheel Drive,Hybrid,21-22,130R White,Ebony,0.0,0.0,1.0,Piazza Acura / Honda of Reading,4.7,4.8,17.0,2111.0,139876.0


---
## PASO 2 — Metadatos de extracción

A cada fila le agregamos 4 columnas. Empiezan con guion bajo `_` para distinguirlas de las columnas originales.

| Columna | Contenido | ¿Para qué sirve? |
|---|---|---|
| `_batch_id` | Número de lote: 1 la primera vez, 2 la siguiente… | Saber en qué ejecución llegó cada fila |
| `_extracted_at` | Fecha y hora de la extracción | Medir la **frescura** de los datos |
| `_source_table` | Tabla de origen (`raw_autos`) | **Lineage**: de dónde viene cada fila |
| `_row_hash` | "Huella digital" del contenido de la fila | **Control de cambios** (PASO 3) |

### La huella (`_row_hash`)

Una **función hash** (aquí SHA-256) convierte un texto en un código corto. Si el texto cambia, aunque sea un carácter, el código cambia por completo. Juntamos los valores de la fila en un texto y le calculamos su hash: **misma fila → misma huella; fila modificada → huella nueva**.

In [5]:
# ── ¿Cómo funciona un hash? ────────────────────────────────────────────────
# El mismo auto con el millaje cambiado en un dígito produce una huella totalmente distinta

print()
for texto in ['Toyota|Camry LE|2019|45000.0',
              'Toyota|Camry LE|2019|45001.0',
              'Toyota|Camry LE|2019|45000.0']:

    huella = hashlib.sha256(texto.encode('utf-8')).hexdigest()[:16]
    print(f'{texto:<30} →  {huella}')
print()


Toyota|Camry LE|2019|45000.0   →  ba39d4514152f776
Toyota|Camry LE|2019|45001.0   →  a7366642a457cd34
Toyota|Camry LE|2019|45000.0   →  ba39d4514152f776



In [6]:
# ── Función para calcular la huella de cada fila ───────────────────────────

def calcular_huella(df):
    '''Regresa la huella (16 caracteres) de cada fila del DataFrame.'''
    texto_fila = df.map(str).agg('|'.join, axis=1)              # 'v1|v2|v3|...'
    return texto_fila.map(lambda t: hashlib.sha256(t.encode('utf-8')).hexdigest()[:16])


# Prueba con las primeras 3 filas
prueba = df_raw.head(3).copy()
prueba['_row_hash'] = calcular_huella(df_raw.head(3))
prueba

,manufacturer,model,year,mileage,engine,transmission,drivetrain,fuel_type,mpg,exterior_color,interior_color,accidents_or_damage,one_owner,personal_use_only,seller_name,seller_rating,driver_rating,driver_reviews_num,price_drop,price,_row_hash
0,Acura,ILX Hybrid 1.5L,2013,92945.0,"1.5L I-4 i-VTEC variable valve control, engine...",Automatic,Front-wheel Drive,Gasoline,39-38,Black,Parchment,0.0,0.0,0.0,Iconic Coach,NaN,4.4,12.0,300.0,13988.0,a5f41d2a94483372
1,Acura,ILX Hybrid 1.5L,2013,61539.0,1.5L I4 8V MPFI SOHC Hybrid,Automatic CVT,Front-wheel Drive,Hybrid,39-38,Silver,NaN,1.0,1.0,1.0,Napleton's Palatine Subaru,4.6,4.4,12.0,NaN,16950.0,d490101e1b4b1443
2,Acura,NSX Base,2017,15904.0,Twin Turbo Gas/Electric V-6 3.5 L/213,Automatic,All-wheel Drive,Hybrid,21-22,130R White,Ebony,0.0,0.0,1.0,Piazza Acura / Honda of Reading,4.7,4.8,17.0,2111.0,139876.0,7497d846a2e6c6e0


---
## PASO 3 — Control de cambios (CDC)

El pipeline no se ejecuta una sola vez: se vuelve a ejecutar cada vez que hay una nueva ingesta. El **control de cambios** evita copiar todo otra vez:

| Tipo de carga | Qué hace |
|---|---|
| **Completa** (*full load*) | Copia toda la tabla. Ocurre la **primera vez**. |
| **Incremental** | Copia **solo las filas nuevas o modificadas**. Ocurre en las siguientes ejecuciones. |

**¿Cómo sabemos qué es nuevo?** Comparamos la huella de cada fila de `raw_autos` con las huellas que **ya están** en `extracted_autos`. Solo pasan las que no estaban.

> Usamos huellas porque `raw_autos` no tiene una columna con la fecha de publicación del anuncio. Si la tuviera, podríamos usar `WHERE fecha > última_fecha_extraída` (estrategia de **marca de agua**).

El estado se guarda en `cdc_autos.json` (último lote, fecha y filas) y cada operación queda en el log `extraccion_log.jsonl`.

In [7]:
# ── Funciones de control de cambios y log ──────────────────────────────────

def leer_cdc():
    '''Lee el estado del control de cambios; vacío si es la primera vez.'''
    if not os.path.exists(CDC_PATH):
        return {}
    with open(CDC_PATH, encoding='utf-8') as f:
        return json.load(f)


def guardar_cdc(estado):
    '''Guarda el estado del control de cambios.'''
    with open(CDC_PATH, 'w', encoding='utf-8') as f:
        json.dump(estado, f, indent=2, ensure_ascii=False)


def escribir_log(filas_leidas, filas_nuevas, tipo, estado='exitoso', error=None):
    '''Agrega UNA línea al log (formato JSON Lines: un JSON por línea).'''
    evento = {
        'timestamp'    : datetime.now().isoformat(timespec='seconds'),
        'tabla_origen' : TABLA_RAW,
        'tabla_destino': TABLA_EXTRACTED,
        'filas_leidas' : filas_leidas,
        'filas_nuevas' : filas_nuevas,
        'tipo'         : tipo,
        'estado'       : estado,
        'error'        : error,
    }
    with open(LOG_PATH, 'a', encoding='utf-8') as f:          # 'a' = agregar al final
        f.write(json.dumps(evento, ensure_ascii=False) + '\n')


def ver_log():
    '''Regresa el historial de operaciones como DataFrame.'''
    if not os.path.exists(LOG_PATH):
        print('Todavía no hay operaciones registradas.')
        return pd.DataFrame()
    return pd.read_json(LOG_PATH, lines=True)


print()
print('Funciones listas')
print()


Funciones listas



In [8]:
# ── La función de extracción ───────────────────────────────────────────────

def extraer():
    '''Extrae raw_autos → extracted_autos, agregando solo las filas nuevas.'''

    try:
        # 1. Leer el origen (solo lectura)
        conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)
        df = pd.read_sql_query(f'SELECT * FROM {TABLA_RAW}', conn)

        # 2. Huellas que ya están en EXTRACTED (si la tabla existe)
        existe = conn.execute(
            "SELECT COUNT(*) FROM sqlite_master WHERE type = 'table' AND name = ?",
            (TABLA_EXTRACTED,)
        ).fetchone()[0] > 0

        if existe:
            huellas_previas = pd.read_sql_query(f'SELECT _row_hash FROM {TABLA_EXTRACTED}', conn)['_row_hash']
        else:
            huellas_previas = pd.Series(dtype=str)
        previas       = set(huellas_previas)                   # huellas distintas (para comparar)
        filas_previas = len(huellas_previas)                   # filas que ya hay (para contar)
        conn.close()

        # 3. Calcular la huella de cada fila y quedarse solo con las nuevas
        df['_row_hash'] = calcular_huella(df)
        nuevas = df[~df['_row_hash'].isin(previas)].copy()

        tipo = 'full_load' if not previas else 'incremental'
        cdc  = leer_cdc()

        # 4. Si hay filas nuevas: metadatos + AGREGAR (append) a extracted_autos
        if len(nuevas) > 0:
            batch_id = cdc.get('ultimo_batch', 0) + 1

            nuevas['_batch_id']     = batch_id
            nuevas['_extracted_at'] = datetime.now().isoformat(timespec='seconds')
            nuevas['_source_table'] = TABLA_RAW

            # Orden: columnas originales y al final los metadatos
            columnas = [c for c in nuevas.columns if not c.startswith('_')]
            nuevas = nuevas[columnas + ['_batch_id', '_extracted_at', '_source_table', '_row_hash']]

            conn = sqlite3.connect(DB_PATH)                    # lectura y escritura
            nuevas.to_sql(TABLA_EXTRACTED, conn, if_exists='append', index=False)
            conn.commit()
            conn.close()

            cdc['ultimo_batch'] = batch_id

        # 5. Actualizar control de cambios y log
        cdc.update({
            'tabla_origen'     : TABLA_RAW,
            'tabla_destino'    : TABLA_EXTRACTED,
            'ultima_extraccion': datetime.now().isoformat(timespec='seconds'),
            'filas_leidas'     : len(df),
            'filas_nuevas'     : len(nuevas),
            'filas_acumuladas' : filas_previas + len(nuevas),
            'tipo'             : tipo,
        })
        guardar_cdc(cdc)
        escribir_log(len(df), len(nuevas), tipo)

        print()
        print(f'{TABLA_RAW} → {TABLA_EXTRACTED}  ({tipo})')
        print(f'  Filas leídas : {len(df):,}')
        print(f'  Filas nuevas : {len(nuevas):,}')
        print()
        return nuevas

    except Exception as e:
        escribir_log(0, 0, 'error', estado='fallido', error=str(e))
        raise                                                  # muestra el error en pantalla


print()
print('extraer() definida')
print()


extraer() definida



> **¿Por qué `if_exists='append'` y no `'replace'`?** `replace` borraría la tabla y la volvería a crear: perderíamos los lotes anteriores. Con `append` las filas nuevas se **agregan al final** y, gracias al control de cambios, nunca se duplican.

---
## PASO 4 — Extracción con control de cambios

Ejecutamos `extraer()` **dos veces** y comparamos:

- **1ª vez:** carga completa (`full_load`): todas las filas son nuevas.
- **2ª vez:** carga incremental (`incremental`): `raw_autos` no cambió, así que **0 filas nuevas**.

### 4.1 Primera ejecución: carga completa

In [9]:
# ── Estado del control de cambios ANTES de extraer ─────────────────────────

print(json.dumps(leer_cdc(), indent=2, ensure_ascii=False))

{}


In [10]:
# ── 1ª ejecución ───────────────────────────────────────────────────────────

df_nuevas = extraer()


raw_autos → extracted_autos  (full_load)
  Filas leídas : 200,000
  Filas nuevas : 200,000



In [11]:
# ── Estado del control de cambios DESPUÉS de la 1ª ejecución ───────────────

print(json.dumps(leer_cdc(), indent=2, ensure_ascii=False))

{
  "ultimo_batch": 1,
  "tabla_origen": "raw_autos",
  "tabla_destino": "extracted_autos",
  "ultima_extraccion": "2026-11-24T11:17:53",
  "filas_leidas": 200000,
  "filas_nuevas": 200000,
  "filas_acumuladas": 200000,
  "tipo": "full_load"
}


### 4.2 Segunda ejecución: carga incremental

`raw_autos` no cambió, así que todas sus huellas ya están en `extracted_autos`: no debe entrar ninguna fila y la tabla no debe crecer.

In [12]:
# ── 2ª ejecución ───────────────────────────────────────────────────────────

df_nuevas_2 = extraer()


raw_autos → extracted_autos  (incremental)
  Filas leídas : 200,000
  Filas nuevas : 0



In [13]:
# ── Estado del control de cambios DESPUÉS de la 2ª ejecución ───────────────

print(json.dumps(leer_cdc(), indent=2, ensure_ascii=False))

{
  "ultimo_batch": 1,
  "tabla_origen": "raw_autos",
  "tabla_destino": "extracted_autos",
  "ultima_extraccion": "2026-11-24T11:19:15",
  "filas_leidas": 200000,
  "filas_nuevas": 0,
  "filas_acumuladas": 200000,
  "tipo": "incremental"
}


---
## PASO 5 — Verificar con SQL

In [14]:
# ── ¿Coinciden RAW y EXTRACTED? ────────────────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

conteos = pd.read_sql_query(f'''
    SELECT (SELECT COUNT(*) FROM {TABLA_RAW})                        AS filas_raw,
           (SELECT COUNT(*) FROM {TABLA_EXTRACTED})                  AS filas_extracted,
           (SELECT COUNT(DISTINCT _row_hash) FROM {TABLA_EXTRACTED}) AS huellas_distintas
''', conn)

conn.close()

print()
print(f"¿Extracción completa (filas RAW = filas EXTRACTED)? {conteos['filas_raw'][0] == conteos['filas_extracted'][0]}")
print()

conteos


¿Extracción completa (filas RAW = filas EXTRACTED)? True



,filas_raw,filas_extracted,huellas_distintas
0,200000,200000,199388


- Si `filas_raw` = `filas_extracted`: la extracción está completa.
- Si `huellas_distintas` es **menor** que `filas_raw`: hay **filas exactamente duplicadas** en `raw_autos` (mismo anuncio repetido). No es un error de la extracción: se anota para limpiarlo en TRANSFORMACION.

In [15]:
# ── ¿Hay filas exactamente duplicadas? ─────────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

duplicadas = pd.read_sql_query(f'''
    SELECT COUNT(*)            AS huellas_repetidas,
           SUM(veces)          AS filas_involucradas,
           SUM(veces) - COUNT(*) AS filas_sobrantes
    FROM (SELECT _row_hash, COUNT(*) AS veces
          FROM {TABLA_EXTRACTED}
          GROUP BY _row_hash
          HAVING COUNT(*) > 1)
''', conn)

conn.close()

duplicadas

,huellas_repetidas,filas_involucradas,filas_sobrantes
0,610,1222,612


In [16]:
# ── ¿Cuántas filas llegaron en cada lote? ──────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

lotes = pd.read_sql_query(f'''
    SELECT _batch_id, _extracted_at, _source_table, COUNT(*) AS filas
    FROM {TABLA_EXTRACTED}
    GROUP BY _batch_id, _extracted_at, _source_table
    ORDER BY _batch_id
''', conn)

conn.close()

lotes

,_batch_id,_extracted_at,_source_table,filas
0,1,2026-11-24T11:17:50,raw_autos,200000


In [17]:
# ── ¿Los metadatos están completos? (no debe haber nulos) ──────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

nulos_metadatos = pd.read_sql_query(f'''
    SELECT COUNT(*)                                                 AS filas,
           SUM(CASE WHEN _batch_id     IS NULL THEN 1 ELSE 0 END)   AS nulos_batch,
           SUM(CASE WHEN _extracted_at IS NULL THEN 1 ELSE 0 END)   AS nulos_fecha,
           SUM(CASE WHEN _source_table IS NULL THEN 1 ELSE 0 END)   AS nulos_origen,
           SUM(CASE WHEN _row_hash     IS NULL THEN 1 ELSE 0 END)   AS nulos_huella
    FROM {TABLA_EXTRACTED}
''', conn)

conn.close()

nulos_metadatos

,filas,nulos_batch,nulos_fecha,nulos_origen,nulos_huella
0,200000,0,0,0,0


In [18]:
# ── Así se ve extracted_autos y cómo quedó la base de datos ────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)
display(pd.read_sql_query(f'SELECT * FROM {TABLA_EXTRACTED} LIMIT 5', conn))
conn.close()

listar_tablas(DB_PATH)

print()
print(f'Tamaño de {os.path.basename(DB_PATH)}: {tamano_mb(DB_PATH):,.1f} MB')
print()

,manufacturer,model,year,mileage,engine,transmission,drivetrain,fuel_type,mpg,exterior_color,interior_color,accidents_or_damage,one_owner,personal_use_only,seller_name,seller_rating,driver_rating,driver_reviews_num,price_drop,price,_batch_id,_extracted_at,_source_table,_row_hash
0,Acura,ILX Hybrid 1.5L,2013,92945.0,"1.5L I-4 i-VTEC variable valve control, engine...",Automatic,Front-wheel Drive,Gasoline,39-38,Black,Parchment,0.0,0.0,0.0,Iconic Coach,NaN,4.4,12.0,300.0,13988.0,1,2026-11-24T11:17:50,raw_autos,a5f41d2a94483372
1,Acura,ILX Hybrid 1.5L,2013,61539.0,1.5L I4 8V MPFI SOHC Hybrid,Automatic CVT,Front-wheel Drive,Hybrid,39-38,Silver,NaN,1.0,1.0,1.0,Napleton's Palatine Subaru,4.6,4.4,12.0,NaN,16950.0,1,2026-11-24T11:17:50,raw_autos,d490101e1b4b1443
2,Acura,NSX Base,2017,15904.0,Twin Turbo Gas/Electric V-6 3.5 L/213,Automatic,All-wheel Drive,Hybrid,21-22,130R White,Ebony,0.0,0.0,1.0,Piazza Acura / Honda of Reading,4.7,4.8,17.0,2111.0,139876.0,1,2026-11-24T11:17:50,raw_autos,7497d846a2e6c6e0
3,Acura,NSX T,1995,320000.0,NaN,5-Speed M/T,Rear-wheel Drive,NaN,NaN,Red,Black,NaN,NaN,NaN,NaN,NaN,5.0,3.0,NaN,75500.0,1,2026-11-24T11:17:50,raw_autos,922931d82e43ace9
4,Acura,NSX Base,2022,1447.0,"3.5L V-6 port/direct injection, DOHC, VTC vari...",Automatic,All-wheel Drive,Gasoline,21-22,Berlina Black,Black,0.0,1.0,1.0,European Exotic Center,NaN,NaN,0.0,5000.0,254995.0,1,2026-11-24T11:17:50,raw_autos,0617942f391c8f2d



  Tablas en: base_datos.db

  TABLA                               CAPA               FILAS   COLS
  ----------------------------------- --------------- -------- ------


  extracted_autos                     extracted_       200,000     24


  raw_autos                           raw_             200,000     20

Tamaño de base_datos.db: 80.9 MB



---
## PASO 6 — Log de operaciones

Cada ejecución de `extraer()` deja una línea en `extraccion_log.jsonl`, incluso si falla (con `estado = 'fallido'` y el error).

In [19]:
# ── Log de operaciones ─────────────────────────────────────────────────────

ver_log()

,timestamp,tabla_origen,tabla_destino,filas_leidas,filas_nuevas,tipo,estado,error
0,2026-11-24 11:17:53,raw_autos,extracted_autos,200000,200000,full_load,exitoso,NaN
1,2026-11-24 11:19:15,raw_autos,extracted_autos,200000,0,incremental,exitoso,NaN


In [20]:
# ── Así se ve el archivo del log por dentro (un JSON por línea) ────────────

with open(LOG_PATH, encoding='utf-8') as f:
    print(f.read())

{"timestamp": "2026-11-24T11:17:53", "tabla_origen": "raw_autos", "tabla_destino": "extracted_autos", "filas_leidas": 200000, "filas_nuevas": 200000, "tipo": "full_load", "estado": "exitoso", "error": null}
{"timestamp": "2026-11-24T11:19:15", "tabla_origen": "raw_autos", "tabla_destino": "extracted_autos", "filas_leidas": 200000, "filas_nuevas": 0, "tipo": "incremental", "estado": "exitoso", "error": null}



---
## PASO 7 — Formatos: CSV vs. JSON vs. Parquet

Muchas fuentes entregan **archivos** en lugar de tablas SQL. Exportamos `extracted_autos` en los tres formatos más comunes y comparamos tamaño y tipos de dato.

| Formato | Cómo guarda los datos | Legible por humanos |
|---|---|---|
| **CSV** | Texto, una fila por línea | Sí |
| **JSON** | Texto con pares `"clave": valor` | Sí |
| **Parquet** | Binario, **por columnas**, comprimido y con tipos de dato | No |

> Los archivos de `Exportaciones/` no se suben al repositorio (el JSON pasa de 100 MB). Se regeneran al ejecutar esta sección.

In [21]:
# ── Leer extracted_autos ───────────────────────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)
df_ext = pd.read_sql_query(f'SELECT * FROM {TABLA_EXTRACTED}', conn)
conn.close()

# SQLite no tiene tipo fecha: _extracted_at llegó como texto. La convertimos a fecha real.
df_ext['_extracted_at'] = pd.to_datetime(df_ext['_extracted_at'])

df_ext.dtypes.to_frame('tipo').T

,manufacturer,model,year,mileage,engine,transmission,drivetrain,fuel_type,mpg,exterior_color,interior_color,accidents_or_damage,one_owner,personal_use_only,seller_name,seller_rating,driver_rating,driver_reviews_num,price_drop,price,_batch_id,_extracted_at,_source_table,_row_hash
tipo,str,str,int64,float64,str,str,str,str,str,str,str,float64,float64,float64,str,float64,float64,float64,float64,float64,int64,datetime64[us],str,str


In [22]:
# ── Exportar en los tres formatos ──────────────────────────────────────────

ruta_csv     = f'{EXPORTS_PATH}/{TABLA_EXTRACTED}.csv'
ruta_json    = f'{EXPORTS_PATH}/{TABLA_EXTRACTED}.json'
ruta_parquet = f'{EXPORTS_PATH}/{TABLA_EXTRACTED}.parquet'

df_ext.to_csv(ruta_csv, index=False)
df_ext.to_json(ruta_json, orient='records', date_format='iso', force_ascii=False)
df_ext.to_parquet(ruta_parquet, index=False)

pd.DataFrame({
    'formato'  : ['CSV', 'JSON', 'Parquet'],
    'tamaño_MB': [round(tamano_mb(r), 2) for r in (ruta_csv, ruta_json, ruta_parquet)],
}).set_index('formato')

,tamaño_MB
formato,
CSV,45.90
JSON,115.00
Parquet,7.68


In [23]:
# ── Leerlos de vuelta: ¿qué tipos de dato se conservan? ────────────────────

desde_csv     = pd.read_csv(ruta_csv, low_memory=False)
desde_json    = pd.read_json(ruta_json, orient='records')
desde_parquet = pd.read_parquet(ruta_parquet)

pd.DataFrame({
    'original': df_ext[COLS_TIPOS].dtypes.astype(str),
    'CSV'     : desde_csv[COLS_TIPOS].dtypes.astype(str),
    'JSON'    : desde_json[COLS_TIPOS].dtypes.astype(str),
    'Parquet' : desde_parquet[COLS_TIPOS].dtypes.astype(str),
})

,original,CSV,JSON,Parquet
year,int64,int64,int64,int64
price,float64,float64,int64,float64
accidents_or_damage,float64,float64,float64,float64
_batch_id,int64,int64,int64,int64
_extracted_at,datetime64[us],str,datetime64[us],datetime64[us]


**Lectura del resultado:**

- **Tamaño:** Parquet es el más pequeño (comprime cada columna). JSON es el más grande porque repite el nombre de cada columna en cada registro.
- **Tipos:** **CSV** pierde la fecha (`_extracted_at` regresa como texto): solo guarda texto y pandas adivina los tipos al leer. **JSON** no tiene tipo fecha; pandas la reconoce porque la columna termina en `_at`, pero es una suposición. Además, JSON no distingue entre `20500.0` y `20500`: `price` regresó como entero (`int64`) en lugar de decimal. **Parquet** guarda el tipo de cada columna dentro del archivo.

| Si necesitas… | Usa |
|---|---|
| Compartir datos para abrir en Excel | **CSV** |
| Intercambiar datos con una API | **JSON** |
| Guardar tablas grandes para análisis | **Parquet** (por eso TRANSFORMACION guarda su resultado en Parquet) |

In [24]:
# ── Liberar memoria ────────────────────────────────────────────────────────

del df_raw, df_ext, desde_csv, desde_json, desde_parquet

print()
print('Memoria liberada')
print()


Memoria liberada



---
### (Opcional) Empezar de cero

Borra `extracted_autos`, su control de cambios y el log para volver a hacer la carga completa. **No toca `raw_autos`.** Quita los `#` para ejecutarla y después vuelve a correr el notebook desde el PASO 1.

In [25]:
# conn = sqlite3.connect(DB_PATH)
# conn.execute(f'DROP TABLE IF EXISTS {TABLA_EXTRACTED}')
# conn.commit()
# conn.execute('VACUUM')                       # libera el espacio en disco del .db
# conn.close()
# for archivo in (CDC_PATH, LOG_PATH):
#     if os.path.exists(archivo):
#         os.remove(archivo)
# print(f'{TABLA_EXTRACTED}, control de cambios y log reiniciados')

---
### Resumen

| Paso | Qué hicimos |
|---|---|
| 1 | Leímos `raw_autos` en **solo lectura** |
| 2 | Definimos los **metadatos** (`_batch_id`, `_extracted_at`, `_source_table`, `_row_hash`) y la huella SHA-256 de cada fila |
| 3 | Definimos el **control de cambios** (`cdc_autos.json`) y el **log** (`extraccion_log.jsonl`) |
| 4 | Extrajimos dos veces: **carga completa** y después **incremental** (0 filas nuevas) |
| 5 | Verificamos con **SQL**: filas RAW vs. EXTRACTED, duplicados, lotes y metadatos completos |
| 6 | Revisamos el **log** de operaciones |
| 7 | Comparamos **CSV, JSON y Parquet** |

**Siguiente notebook:** `TRANSFORMACION.ipynb`, que leerá **`extracted_autos`** (nunca `raw_autos`) para limpiar los datos.